In [2]:
# Paired t-test: REMNET versus standalone CNNs (SUSTAINABILITY ONLY)
# Based on the structure of the supplied Paired_t_test_co2.ipynb.
# Data: original Stage22 sustainability CodeCarbon logs, five seeds.
# REMNET training = sum of three constituent training sessions WITHIN its
# own sustainability notebook; inference = monitored REMNET ensemble session.
# The other models use their independent sustainability experiments.
# No classification outcomes or classification checkpoints are used.

import scipy.stats as stats
import numpy as np
import math
import csv
from pathlib import Path

SEEDS = [7, 49, 112, 345, 476]
n = len(SEEDS)
alpha = 0.05
CARBON_FACTOR = 0.475  # kg CO2e/kWh; standardized assumption, NOT native CodeCarbon emission

# Full-precision measurements (in the same seed order) taken from the
# original CodeCarbon logs; verified against the standalone energy summaries.
# Total = training + inference; REMNET training sums all three branches.
metrics_data = {'CO2 (kg; fixed factor 0.475)': {'ResNet-50': [0.007988823830711025, 0.008074054368807728, 0.008094522978707096, 0.008076308015290554, 0.00805540552329069], 'MobileNetV2': [0.003524442318609397, 0.0035575125256850984, 0.0035286485972518848, 0.003532557150637637, 0.003535704967174016], 'EfficientNet-Lite0': [0.003817884940745782, 0.0038850195398632457, 0.0038797918947074132, 0.0038720018794211743, 0.0038835392697040925], 'REMNET': [0.0024070355385966313, 0.002055015300858375, 0.00197575584620993, 0.001981105552279359, 0.002076039902111083]}, 'Energy (kWh; total)': {'ResNet-50': [0.01681857648570742, 0.016998009197489953, 0.017041101007804413, 0.017002753716401166, 0.016958748470085664], 'MobileNetV2': [0.0074198785654934684, 0.007489500054073892, 0.007428733888951337, 0.007436962422395025, 0.007443589404576876], 'EfficientNet-Lite0': [0.008037652506833226, 0.008178988504975254, 0.008167982936226134, 0.008151582904044578, 0.008175872146745458], 'REMNET': [0.005067443239150803, 0.0043263480018071055, 0.004159485992020906, 0.00417074853111444, 0.004370610320233859]}}

model_order = ["ResNet-50", "MobileNetV2", "EfficientNet-Lite0"]
metric_order = ["CO2 (kg; fixed factor 0.475)", "Energy (kWh; total)"]

OUT = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
OUT.mkdir(parents=True, exist_ok=True)
result_file = OUT / 'REMNET_Sustainability_Paired_T_Test_Results.csv'

# Holm-Bonferroni adjusted p-values, step-down with monotone constraint.
def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    indexes = np.argsort(p_values)
    result = np.empty(len(p_values), dtype=float)
    running = 0.0
    for rank, index in enumerate(indexes):
        running = max(running, (len(p_values) - rank) * p_values[index])
        result[index] = min(1.0, running)
    return result

def format_to_unicode_power(value):
    if value == 0:
        return '0'
    exponent = math.floor(math.log10(abs(value)))
    mantissa = value / (10 ** exponent)
    superscripts = str.maketrans('0123456789-', '⁰¹²³⁴⁵⁶⁷⁸⁹⁻')
    return f"{mantissa:.2f} × 10{str(exponent).translate(superscripts)}"

records = []
for metric in metric_order:
    remnet = np.array(metrics_data[metric]['REMNET'], dtype=float)
    assert len(remnet) == n and np.isfinite(remnet).all()
    for model in model_order:
        baseline = np.array(metrics_data[metric][model], dtype=float)
        assert len(baseline) == n and np.isfinite(baseline).all()
        # Paired seed differences. Negative means lower measured energy/CO2 for REMNET.
        differences = remnet - baseline
        sd_diff = float(np.std(differences, ddof=1))
        if sd_diff == 0:
            raise ValueError(f'Cannot compute paired t/effect size: zero SD of differences for {metric}, {model}')
        t_stat, p_value = stats.ttest_rel(remnet, baseline, alternative='two-sided')
        mean_diff = float(np.mean(differences))
        sem_diff = float(stats.sem(differences))
        ci_low, ci_high = stats.t.interval(0.95, df=n-1, loc=mean_diff, scale=sem_diff)
        records.append(dict(
            Metric=metric, Comparison=f'REMNET vs {model}', Model=model,
            n_pairs=n, degrees_of_freedom=n-1,
            REMNET_mean=float(remnet.mean()), REMNET_sample_SD=float(remnet.std(ddof=1)),
            CNN_mean=float(baseline.mean()), CNN_sample_SD=float(baseline.std(ddof=1)),
            Paired_mean_difference_REMNET_minus_CNN=mean_diff,
            Paired_difference_sample_SD=sd_diff,
            CI95_lower=float(ci_low), CI95_upper=float(ci_high),
            Paired_Cohens_dz=float(mean_diff / sd_diff),
            Paired_t_statistic=float(t_stat), P_value=float(p_value),
            Holm_adjusted_p_value_within_metric=float('nan'),
            Holm_adjusted_p_value_all_6_tests=float('nan'),
            Significant_after_Holm_within_metric='',
            Difference_direction=('REMNET lower' if mean_diff < 0 else 'REMNET higher'),
            Seed_order='7,49,112,345,476',
            Data_source='Sustainability only (independent CodeCarbon experiments)',
            CO2_method='Standardized fixed-factor CO2 = energy_kWh * 0.475; not original CodeCarbon emissions',
        ))

for metric in metric_order:
    indices = [i for i, r in enumerate(records) if r['Metric'] == metric]
    adjusted = holm_adjust([records[i]['P_value'] for i in indices])
    for i, p in zip(indices, adjusted):
        records[i]['Holm_adjusted_p_value_within_metric'] = float(p)
        records[i]['Significant_after_Holm_within_metric'] = ('Yes' if p < alpha else 'No')
for i, p in enumerate(holm_adjust([r['P_value'] for r in records])):
    records[i]['Holm_adjusted_p_value_all_6_tests'] = float(p)

# CSV: machine-readable full-precision outcomes; notebook prints readable summary.
with open(result_file, 'w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=list(records[0].keys()))
    writer.writeheader()
    writer.writerows(records)

print('Sustainability paired t-tests: REMNET vs individual CNNs')
print(f'Paired seeds: {SEEDS}; n={n}; df={n-1}; alpha={alpha}; two-sided')
print('Difference = REMNET − CNN; negative indicates lower measured energy/CO2 for REMNET')
print('Cohen dz = paired mean difference / sample SD of paired differences')
print('Holm adjustment applies to three CNN comparisons within each metric.')
print('The CSV also includes a more conservative across-all-six-tests correction.')
print('IMPORTANT: CO2 here is a fixed scalar multiple of energy, not an independent statistical outcome.')
print('These are independent sustainability experiments aligned by seed ID; not shared trained checkpoints.')
for metric in metric_order:
    print('\n' + metric)
    print(f"{'Model':<21} {'REMNET mean±SD':>23} {'CNN mean±SD':>23} {'Mean difference':>17} {'95% CI':>27} {'dz':>10} {'t':>10} {'p':>16} {'Holm p':>16}")
    print('-' * 173)
    for r in [x for x in records if x['Metric'] == metric]:
        fmt=lambda a,b: f'{a:.9f} ± {b:.9f}'
        print(f"{r['Model']:<21} {fmt(r['REMNET_mean'], r['REMNET_sample_SD']):>23} {fmt(r['CNN_mean'], r['CNN_sample_SD']):>23} {r['Paired_mean_difference_REMNET_minus_CNN']:>17.9f} {('(' + format(r['CI95_lower'], '.9f') + ', ' + format(r['CI95_upper'], '.9f') + ')'):>27} {r['Paired_Cohens_dz']:>10.3f} {r['Paired_t_statistic']:>10.3f} {format_to_unicode_power(r['P_value']):>16} {format_to_unicode_power(r['Holm_adjusted_p_value_within_metric']):>16}")

print('\nSaved:', result_file)
print('Original CodeCarbon estimates (emissions) are distinct from the standardized fixed-factor CO2 tested here.')


Sustainability paired t-tests: REMNET vs individual CNNs
Paired seeds: [7, 49, 112, 345, 476]; n=5; df=4; alpha=0.05; two-sided
Difference = REMNET − CNN; negative indicates lower measured energy/CO2 for REMNET
Cohen dz = paired mean difference / sample SD of paired differences
Holm adjustment applies to three CNN comparisons within each metric.
The CSV also includes a more conservative across-all-six-tests correction.
IMPORTANT: CO2 here is a fixed scalar multiple of energy, not an independent statistical outcome.
These are independent sustainability experiments aligned by seed ID; not shared trained checkpoints.

CO2 (kg; fixed factor 0.475)
Model                          REMNET mean±SD             CNN mean±SD   Mean difference                      95% CI         dz          t                p           Holm p
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Re